In [2]:
import copy
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import pyreadstat
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import flwr as fl
from flwr.simulation import start_simulation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

## Load and audit the original NDHS data

In [8]:
# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

DATA_PATH = Path(r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

OUTPUT_DIR = Path("NDHS_FL_Research")
AUDIT_DIR = OUTPUT_DIR / "01_data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. LOAD ORIGINAL DATA
# ------------------------------------------------------------

df, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    apply_value_formats=False
)

print("DATASET LOADED")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of observations:", len(df))
print("Number of variables:", len(df.columns))

# ------------------------------------------------------------
# 3. REQUIRED VARIABLE CHECK
# ------------------------------------------------------------

required_variables = [
    "m15_1",
    "v024",
    "v001",
    "v021",
    "v005",
    "v022",
    "v023",
    "v501",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01",
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131",
    "b3_01",
    "v011"
]

missing_variables = [
    col for col in required_variables
    if col not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Required variables missing from dataset: {missing_variables}"
    )

print("\nAll required variables are present.")

# ------------------------------------------------------------
# 4. RECONSTRUCT THE BINARY OUTCOME
# ------------------------------------------------------------

facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["m15_1"] = pd.to_numeric(df["m15_1"], errors="coerce")

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

# Keep only observations with a valid binary outcome.
model_df = df.loc[
    df["facility_delivery"].notna()
].copy()

model_df["facility_delivery"] = (
    model_df["facility_delivery"].astype(int)
)

print("\nOUTCOME RECONSTRUCTION")
print("=" * 60)

print("Original observations:", len(df))
print("Eligible observations:", len(model_df))
print("Excluded observations:", len(df) - len(model_df))

print("\nOutcome distribution:")
print(
    model_df["facility_delivery"]
    .value_counts()
    .sort_index()
)

print("\nOutcome percentages:")
print(
    model_df["facility_delivery"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 5. CHECK REGIONAL DISTRIBUTION
# ------------------------------------------------------------

region_summary = (
    model_df.groupby("v024")
    .agg(
        sample_size=("facility_delivery", "size"),
        facility_deliveries=("facility_delivery", "sum"),
        facility_prevalence=("facility_delivery", "mean")
    )
    .reset_index()
)

region_summary["facility_prevalence"] *= 100

print("\nREGIONAL DISTRIBUTION")
print("=" * 60)
print(region_summary.to_string(index=False))

region_summary.to_csv(
    AUDIT_DIR / "regional_outcome_distribution.csv",
    index=False
)

# ------------------------------------------------------------
# 6. VERIFY PSU AND CLUSTER IDENTIFIERS
# ------------------------------------------------------------

print("\nPSU AND CLUSTER CHECKS")
print("=" * 60)

print("v001 missing:", df["v001"].isna().sum())
print("v021 missing:", df["v021"].isna().sum())

same_cluster = (
    df["v001"].astype("string")
    == df["v021"].astype("string")
)

print("v001 equals v021 for all nonmissing pairs:",
      same_cluster[df["v001"].notna() & df["v021"].notna()].all())

print("Unique v001:", df["v001"].nunique())
print("Unique v021:", df["v021"].nunique())

print(
    "Unique PSUs in eligible sample:",
    model_df["v021"].nunique()
)

# Check whether a PSU is assigned to more than one region.
psu_zone_counts = (
    model_df.groupby("v021")["v024"]
    .nunique()
)

print(
    "PSUs appearing in multiple regions:",
    int((psu_zone_counts > 1).sum())
)

# ------------------------------------------------------------
# 7. SURVEY WEIGHT AND STRATA CHECKS
# ------------------------------------------------------------

model_df["weight"] = (
    pd.to_numeric(model_df["v005"], errors="coerce") / 1_000_000
)

print("\nSURVEY DESIGN CHECKS")
print("=" * 60)

print("Missing weights:", model_df["weight"].isna().sum())
print("Non-positive weights:", (model_df["weight"] <= 0).sum())
print("Unique PSUs:", model_df["v021"].nunique())
print("Unique v022 strata:", model_df["v022"].nunique())
print("Unique v023 values:", model_df["v023"].nunique())

if model_df["weight"].isna().any():
    raise ValueError("Missing survey weights found in eligible sample.")

if (model_df["weight"] <= 0).any():
    raise ValueError("Non-positive survey weights found.")

# Weighted and unweighted outcome prevalence.
unweighted_prevalence = model_df["facility_delivery"].mean()

weighted_prevalence = np.average(
    model_df["facility_delivery"],
    weights=model_df["weight"]
)

print("\nUnweighted facility prevalence:",
      round(unweighted_prevalence * 100, 3), "%")

print("Weighted facility prevalence:",
      round(weighted_prevalence * 100, 3), "%")

# Regional weighted prevalence.
weighted_region_rows = []

for zone, group in model_df.groupby("v024"):

    weighted_rate = np.average(
        group["facility_delivery"],
        weights=group["weight"]
    )

    weighted_region_rows.append({
        "v024": zone,
        "sample_size": len(group),
        "unweighted_prevalence": group["facility_delivery"].mean(),
        "weighted_prevalence": weighted_rate
    })

weighted_region_summary = pd.DataFrame(weighted_region_rows)

weighted_region_summary[
    ["unweighted_prevalence", "weighted_prevalence"]
] *= 100

weighted_region_summary.to_csv(
    AUDIT_DIR / "weighted_regional_prevalence.csv",
    index=False
)

print("\nWeighted regional prevalence:")
print(weighted_region_summary.to_string(index=False))

# ------------------------------------------------------------
# 8. INSPECT v701 VALUE LABELS AND MISSINGNESS
# ------------------------------------------------------------

print("\nPARTNER EDUCATION: v701")
print("=" * 60)

print("Variable label:")
print(meta.column_names_to_labels.get("v701", "No label available"))

print("\nValue labels:")
print(meta.variable_value_labels.get("v701", "No value labels available"))

print("\nObserved v701 frequencies, including missing:")
print(
    model_df["v701"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissing v701 by marital status (v501):")

v701_missing_by_marital = (
    model_df.assign(v701_missing=model_df["v701"].isna())
    .groupby("v501", dropna=False)
    .agg(
        sample_size=("v701_missing", "size"),
        missing_v701=("v701_missing", "sum")
    )
    .reset_index()
)

v701_missing_by_marital["missing_percentage"] = (
    100 * v701_missing_by_marital["missing_v701"]
    / v701_missing_by_marital["sample_size"]
)

print(v701_missing_by_marital.to_string(index=False))

v701_missing_by_marital.to_csv(
    AUDIT_DIR / "v701_missingness_by_marital_status.csv",
    index=False
)

# ------------------------------------------------------------
# 9. FEATURE MISSINGNESS AUDIT
# ------------------------------------------------------------

baseline_features = [
    "age_at_birth",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01"
]

expanded_features = baseline_features + [
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131"
]

# Derive age at birth.
model_df["age_at_birth"] = np.floor(
    (pd.to_numeric(model_df["b3_01"], errors="coerce")
     - pd.to_numeric(model_df["v011"], errors="coerce")) / 12
)

feature_missingness = pd.DataFrame({
    "feature": expanded_features,
    "missing_n": [
        model_df[col].isna().sum()
        for col in expanded_features
    ],
    "missing_percentage": [
        model_df[col].isna().mean() * 100
        for col in expanded_features
    ]
})

print("\nFEATURE MISSINGNESS")
print("=" * 60)
print(feature_missingness.to_string(index=False))

feature_missingness.to_csv(
    AUDIT_DIR / "feature_missingness.csv",
    index=False
)

# ------------------------------------------------------------
# 10. SAVE AUDIT SUMMARY
# ------------------------------------------------------------

audit_summary = {
    "source_file": DATA_PATH.name,
    "original_rows": int(len(df)),
    "original_columns": int(len(df.columns)),
    "eligible_rows": int(len(model_df)),
    "home_deliveries": int(
        (model_df["facility_delivery"] == 0).sum()
    ),
    "facility_deliveries": int(
        (model_df["facility_delivery"] == 1).sum()
    ),
    "unweighted_facility_prevalence": float(
        unweighted_prevalence
    ),
    "weighted_facility_prevalence": float(
        weighted_prevalence
    ),
    "unique_psus": int(model_df["v021"].nunique()),
    "unique_v022_strata": int(model_df["v022"].nunique()),
    "v001_equals_v021": bool(
        same_cluster[
            df["v001"].notna() & df["v021"].notna()
        ].all()
    )
}

with open(
    AUDIT_DIR / "audit_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit_summary, f, indent=4)

print("\nSTEP 1 COMPLETED")
print("Audit outputs saved to:", AUDIT_DIR.resolve())

DATASET LOADED
Dataset shape: (41821, 5394)
Number of observations: 41821
Number of variables: 5394

All required variables are present.

OUTCOME RECONSTRUCTION
Original observations: 41821
Eligible observations: 21442
Excluded observations: 20379

Outcome distribution:
facility_delivery
0    12441
1     9001
Name: count, dtype: int64

Outcome percentages:
facility_delivery
0    58.02
1    41.98
Name: proportion, dtype: float64

REGIONAL DISTRIBUTION
 v024  sample_size  facility_deliveries  facility_prevalence
    1         3838                 1997            52.032308
    2         4498                 1169            25.989329
    3         6307                  989            15.680989
    4         2338                 1887            80.710009
    5         2050                  998            48.682927
    6         2411                 1961            81.335545

PSU AND CLUSTER CHECKS
v001 missing: 0
v021 missing: 0
v001 equals v021 for all nonmissing pairs: True
Unique v001: 1